# Hybrid Retrieval Pipeline — Snowflake Arctic + BM25 + RRF + Reranker

End-to-end hybrid retrieval for the DevRev Search benchmark.

| Stage | Technique | Details |
|-------|-----------|----------|
| Dense | Snowflake Arctic Embed L v2.0 | 1024d, FAISS FlatIP |
| Sparse | BM25 (Okapi) | Tokenized corpus |
| Fusion | Reciprocal Rank Fusion (RRF) | k=60 |
| Reranking | BGE-reranker-v2-m3 | Cross-encoder, top-100 candidates |

**Prerequisites:** Run `python snowflake_embedding.py` (or copy `embeddings_local.npy`, `bm25_local.pkl`, `faiss_index_local/` from bench).

In [11]:
import os, json, pickle, re, time, warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import faiss
from rank_bm25 import BM25Okapi
from tqdm import tqdm
from datasets import load_dataset

warnings.filterwarnings('ignore')

EMBED_MODEL_NAME = "Snowflake/snowflake-arctic-embed-l-v2.0"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
INDEX_DIR = "faiss_index_local"

## 1. Load Data, Indexes & Models

In [12]:
# Load FAISS index and doc mapping
index = faiss.read_index(os.path.join(INDEX_DIR, "knowledge_base_flat.index"))
with open(os.path.join(INDEX_DIR, "doc_mapping.pkl"), "rb") as f:
    mapping = pickle.load(f)

doc_ids = mapping["doc_ids"]
documents = mapping["documents"]
doc_titles = mapping["doc_titles"]
doc_texts = mapping["doc_texts"]

print(f"FAISS index: {index.ntotal:,} vectors (dim={index.d})")
print(f"Documents: {len(documents):,}")

FAISS index: 65,224 vectors (dim=1024)
Documents: 65,224


In [13]:
# Load datasets
test_queries = load_dataset("devrev/search", "test_queries", split="test")
annotated_queries = load_dataset("devrev/search", "annotated_queries", split="train")

print(f"Test queries: {len(test_queries)}")
print(f"Annotated queries (eval): {len(annotated_queries)}")

Test queries: 92
Annotated queries (eval): 291


In [14]:
# Load embedding model for query encoding
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=device, trust_remote_code=True)
print(f"Embedding model loaded: dim={embed_model.get_sentence_embedding_dimension()}")

Device: mps


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 21874.18it/s]


Embedding model loaded: dim=1024


In [15]:
# Load BM25 index
with open("bm25_local.pkl", "rb") as f:
    bm25 = pickle.load(f)
print(f"BM25 index loaded: {bm25.corpus_size:,} docs")

BM25 index loaded: 65,224 docs


In [16]:
# Load reranker
print(f"Loading reranker: {RERANKER_MODEL_NAME} on {device}")
reranker = CrossEncoder(RERANKER_MODEL_NAME, device=device, trust_remote_code=True)
print("Reranker loaded.")

Loading reranker: BAAI/bge-reranker-v2-m3 on mps


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 7881.28it/s]


Reranker loaded.


## 2. Core Retrieval Functions

In [17]:
def tokenize(text: str) -> list[str]:
    return re.findall(r'\w+', text.lower())


def embed_queries(queries: list[str]) -> np.ndarray:
    """Embed queries with the snowflake prefix."""
    texts = [QUERY_PREFIX + q for q in queries]
    embs = embed_model.encode(texts, batch_size=64, show_progress_bar=False,
                               convert_to_numpy=True, normalize_embeddings=True)
    return embs.astype(np.float32)


def embed_single(query: str) -> np.ndarray:
    return embed_queries([query])[0]


def search_dense(query_emb: np.ndarray, k: int = 100) -> list[tuple[int, float]]:
    """FAISS dense search. Returns [(doc_idx, score), ...]."""
    q = query_emb.reshape(1, -1)
    scores, indices = index.search(q, k)
    return [(int(i), float(s)) for s, i in zip(scores[0], indices[0]) if i >= 0]


def search_bm25(query: str, k: int = 100) -> list[tuple[int, float]]:
    """BM25 sparse search. Returns [(doc_idx, score), ...]."""
    tokens = tokenize(query)
    scores = bm25.get_scores(tokens)
    top_k = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in top_k if scores[i] > 0]


# Quick test
q_emb = embed_single("How do I set up AirSync?")
d_results = search_dense(q_emb, k=5)
b_results = search_bm25("How do I set up AirSync?", k=5)
print("Dense top-3:")
for idx, score in d_results[:3]:
    print(f"  {score:.4f} | {doc_ids[idx]} | {doc_titles[idx][:60]}")
print("BM25 top-3:")
for idx, score in b_results[:3]:
    print(f"  {score:.2f} | {doc_ids[idx]} | {doc_titles[idx][:60]}")

Dense top-3:
  0.4619 | ART-16805_KNOWLEDGE_NODE-0 | Articulate Reach 360 AirSync | AirSync | Snap-ins | DevRev
  0.4522 | ART-16805_KNOWLEDGE_NODE-6 | Articulate Reach 360 AirSync | AirSync | Snap-ins | DevRev
  0.4509 | ART-2044_KNOWLEDGE_NODE-0 | ServiceNow AirSync | AirSync | Snap-ins | DevRev
BM25 top-3:
  24.39 | ART-4169_KNOWLEDGE_NODE-9 | DevRev For Startups
  23.39 | ART-973_KNOWLEDGE_NODE-2 | About
  21.58 | ART-2819_KNOWLEDGE_NODE-27 | Exotel | Integrate | Snap-ins | DevRev


## 3. Reciprocal Rank Fusion (RRF)

Merges dense + BM25 ranked lists. Each source contributes `1/(k+rank+1)` per document.

In [18]:
def rrf_fuse(*ranked_lists, k: int = 60) -> list[tuple[int, float]]:
    """Reciprocal Rank Fusion across multiple ranked lists."""
    rrf_scores = defaultdict(float)
    for rlist in ranked_lists:
        for rank, (doc_idx, _) in enumerate(rlist):
            rrf_scores[doc_idx] += 1.0 / (k + rank + 1)
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)


# Quick test
fused = rrf_fuse(d_results, b_results)
print(f"RRF fused: {len(fused)} unique docs from dense + BM25")
for idx, score in fused[:5]:
    print(f"  RRF={score:.4f} | {doc_ids[idx]} | {doc_titles[idx][:60]}")

RRF fused: 10 unique docs from dense + BM25
  RRF=0.0164 | ART-16805_KNOWLEDGE_NODE-0 | Articulate Reach 360 AirSync | AirSync | Snap-ins | DevRev
  RRF=0.0164 | ART-4169_KNOWLEDGE_NODE-9 | DevRev For Startups
  RRF=0.0161 | ART-16805_KNOWLEDGE_NODE-6 | Articulate Reach 360 AirSync | AirSync | Snap-ins | DevRev
  RRF=0.0161 | ART-973_KNOWLEDGE_NODE-2 | About
  RRF=0.0159 | ART-2044_KNOWLEDGE_NODE-0 | ServiceNow AirSync | AirSync | Snap-ins | DevRev


## 4. Cross-Encoder Reranking

Reranks top-100 RRF candidates using BGE-reranker-v2-m3. This is the biggest lever for precision.

In [19]:
def rerank_candidates(query: str, candidates: list[tuple[int, float]], top_k: int = 50) -> list[tuple[int, float]]:
    """Rerank candidates using cross-encoder. Truncates docs to 512 chars."""
    if not candidates:
        return []
    candidates = candidates[:100]
    pairs = [(query, documents[idx][:512]) for idx, _ in candidates]
    scores = reranker.predict(pairs, show_progress_bar=False)
    scored = [(candidates[i][0], float(scores[i])) for i in range(len(candidates))]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]


# Quick test
fused_full = rrf_fuse(search_dense(q_emb, k=100), search_bm25("How do I set up AirSync?", k=100))
reranked = rerank_candidates("How do I set up AirSync?", fused_full, top_k=5)
print("Reranked results:")
for idx, score in reranked:
    print(f"  {score:.4f} | {doc_ids[idx]} | {doc_titles[idx][:60]}")

Reranked results:
  0.9950 | ART-2045_KNOWLEDGE_NODE-29 | AirSync | Snap-ins | DevRev
  0.8739 | ART-2045_KNOWLEDGE_NODE-60 | AirSync | Snap-ins | DevRev
  0.8120 | ART-16263_KNOWLEDGE_NODE-32 | Confluence Datacenter AirSync | AirSync | Snap-ins | DevRev
  0.7815 | ART-4274_KNOWLEDGE_NODE-36 | Figma AirSync | AirSync | Snap-ins | DevRev
  0.7716 | ART-16804_KNOWLEDGE_NODE-24 | Google Calendar AirSync | Integrate | Snap-ins | DevRev


## 5. Full Hybrid Pipeline

In [20]:
def hybrid_retrieve(
    query: str,
    top_k: int = 50,
    dense_k: int = 100,
    bm25_k: int = 100,
    use_bm25: bool = True,
    use_reranking: bool = True,
) -> list[tuple[int, float]]:
    """Dense + BM25 -> RRF -> Rerank."""
    q_emb = embed_single(query)
    dense_results = search_dense(q_emb, k=dense_k)

    if use_bm25:
        bm25_results = search_bm25(query, k=bm25_k)
        fused = rrf_fuse(dense_results, bm25_results)
    else:
        fused = dense_results

    if use_reranking:
        return rerank_candidates(query, fused, top_k=top_k)
    return fused[:top_k]


# Test
results = hybrid_retrieve("end customer organization name not appearing in ticket", top_k=5)
print("Hybrid results:")
for rank, (idx, score) in enumerate(results):
    print(f"  [{rank+1}] {score:.4f} | {doc_ids[idx]} | {doc_titles[idx][:60]}")

Hybrid results:
  [1] 0.5847 | ART-1953_KNOWLEDGE_NODE-29 | Customer email notifications | Computer by DevRev | DevRev
  [2] 0.2873 | ART-1978_KNOWLEDGE_NODE-44 | Customer portal | Computer for Support Teams | DevRev
  [3] 0.1663 | ART-1986_KNOWLEDGE_NODE-45 | Service-level agreement | Computer for Support Teams | DevRe
  [4] 0.1118 | ART-1981_KNOWLEDGE_NODE-34 | Support best practices | Computer for Support Teams | DevRev
  [5] 0.0309 | ART-2040_KNOWLEDGE_NODE-27 | Zendesk AirSync | AirSync | Snap-ins | DevRev


## 6. Evaluation Engine

In [21]:
def evaluate_pipeline(
    pipeline_fn,
    annotated_data,
    top_k_list=(10, 50),
    max_queries: int = None,
    label: str = "Pipeline",
) -> dict:
    """Evaluate retrieval pipeline on annotated queries."""
    items = list(annotated_data)
    if max_queries:
        items = items[:max_queries]

    max_k = max(top_k_list)
    results_by_k = {k: {"recalls": [], "precisions": [], "mrrs": []} for k in top_k_list}
    per_query = []

    for item in tqdm(items, desc=f"Eval: {label}"):
        query = item["query"]
        golden_ids = {r["id"] for r in item["retrievals"]}

        ranked = pipeline_fn(query, top_k=max_k)
        retrieved_ids = [doc_ids[idx] for idx, _ in ranked]

        for k in top_k_list:
            top_ids = retrieved_ids[:k]
            hits = sum(1 for rid in top_ids if rid in golden_ids)
            recall = hits / len(golden_ids) if golden_ids else 0
            precision = hits / k
            results_by_k[k]["recalls"].append(recall)
            results_by_k[k]["precisions"].append(precision)

            rr = 0.0
            for rank_pos, rid in enumerate(top_ids):
                if rid in golden_ids:
                    rr = 1.0 / (rank_pos + 1)
                    break
            results_by_k[k]["mrrs"].append(rr)

        per_query.append({
            "query_id": item["query_id"], "query": query,
            "golden_count": len(golden_ids),
            "hits@10": sum(1 for rid in retrieved_ids[:10] if rid in golden_ids),
            "hits@50": sum(1 for rid in retrieved_ids[:50] if rid in golden_ids),
        })

    print(f"\n{'='*70}")
    print(f"  {label} -- {len(items)} queries")
    print(f"{'='*70}")
    metrics = {"label": label, "n_queries": len(items)}
    for k in top_k_list:
        r = np.mean(results_by_k[k]["recalls"]) * 100
        p = np.mean(results_by_k[k]["precisions"]) * 100
        m = np.mean(results_by_k[k]["mrrs"])
        print(f"  Recall@{k}: {r:.2f}%   Precision@{k}: {p:.2f}%   MRR@{k}: {m:.4f}")
        metrics[f"recall@{k}"] = r
        metrics[f"precision@{k}"] = p
        metrics[f"mrr@{k}"] = m

    metrics["per_query"] = per_query
    return metrics

print("Evaluation engine ready.")

Evaluation engine ready.


## 7. Ablation Study

| # | Config | What it tests |
|---|--------|---------------|
| 1 | Dense only | Snowflake embedding quality |
| 2 | BM25 only | Sparse baseline |
| 3 | Dense + BM25 (RRF) | Hybrid fusion value |
| 4 | Dense + BM25 + Reranker | Full pipeline |

In [22]:
# Pipeline variants

def pipe_dense_only(query, top_k=50):
    q_emb = embed_single(query)
    return search_dense(q_emb, k=top_k)

def pipe_bm25_only(query, top_k=50):
    return search_bm25(query, k=top_k)

def pipe_dense_bm25_rrf(query, top_k=50):
    q_emb = embed_single(query)
    dense = search_dense(q_emb, k=100)
    sparse = search_bm25(query, k=100)
    return rrf_fuse(dense, sparse)[:top_k]

def pipe_hybrid_rerank(query, top_k=50):
    return hybrid_retrieve(query, top_k=top_k,
                           use_bm25=True, use_reranking=True)

In [ ]:
ablation = {}

# 1. Dense only
ablation["dense"] = evaluate_pipeline(pipe_dense_only, annotated_queries, label="1. Dense Only (Snowflake)")

# 2. BM25 only
ablation["bm25"] = evaluate_pipeline(pipe_bm25_only, annotated_queries, label="2. BM25 Only")

Eval: 1. Dense Only (Snowflake): 100%|██████████| 291/291 [00:05<00:00, 55.99it/s]



  1. Dense Only (Snowflake) -- 291 queries
  Recall@10: 4.85%   Precision@10: 3.92%   MRR@10: 0.0959
  Recall@50: 9.16%   Precision@50: 2.48%   MRR@50: 0.1010


Eval: 2. BM25 Only: 100%|██████████| 291/291 [00:20<00:00, 14.48it/s]


  2. BM25 Only -- 291 queries
  Recall@10: 14.63%   Precision@10: 10.17%   MRR@10: 0.2756
  Recall@50: 28.38%   Precision@50: 4.65%   MRR@50: 0.2847


In [24]:
# 3. Dense + BM25 RRF
ablation["rrf"] = evaluate_pipeline(pipe_dense_bm25_rrf, annotated_queries, label="3. Dense + BM25 (RRF)")

Eval: 3. Dense + BM25 (RRF): 100%|██████████| 291/291 [00:27<00:00, 10.67it/s]


  3. Dense + BM25 (RRF) -- 291 queries
  Recall@10: 12.56%   Precision@10: 9.45%   MRR@10: 0.2330
  Recall@50: 27.67%   Precision@50: 4.77%   MRR@50: 0.2444


In [25]:
# 4. Hybrid + Reranker (slower -- cross-encoder on 100 candidates per query)
ablation["rerank"] = evaluate_pipeline(pipe_hybrid_rerank, annotated_queries, label="4. Dense + BM25 + Reranker")

Eval: 4. Dense + BM25 + Reranker: 100%|██████████| 291/291 [14:48<00:00,  3.05s/it]


  4. Dense + BM25 + Reranker -- 291 queries
  Recall@10: 20.66%   Precision@10: 14.81%   MRR@10: 0.4272
  Recall@50: 31.23%   Precision@50: 5.88%   MRR@50: 0.4315


In [26]:
# Ablation summary
print("\n" + "=" * 95)
print("ABLATION SUMMARY")
print("=" * 95)
print(f"{'Method':<40} {'R@10':>8} {'R@50':>8} {'P@10':>8} {'P@50':>8} {'MRR@10':>8}")
print("-" * 95)
for key, m in ablation.items():
    print(f"{m['label']:<40} {m.get('recall@10',0):>7.2f}% {m.get('recall@50',0):>7.2f}% "
          f"{m.get('precision@10',0):>7.2f}% {m.get('precision@50',0):>7.2f}% "
          f"{m.get('mrr@10',0):>8.4f}")
print("=" * 95)


ABLATION SUMMARY
Method                                       R@10     R@50     P@10     P@50   MRR@10
-----------------------------------------------------------------------------------------------
1. Dense Only (Snowflake)                   4.85%    9.16%    3.92%    2.48%   0.0959
2. BM25 Only                               14.63%   28.38%   10.17%    4.65%   0.2756
3. Dense + BM25 (RRF)                      12.56%   27.67%    9.45%    4.77%   0.2330
4. Dense + BM25 + Reranker                 20.66%   31.23%   14.81%    5.88%   0.4272


## 8. Error Analysis

In [27]:
# Analyze worst-performing queries from the best pipeline
best_key = max(ablation, key=lambda k: ablation[k].get("recall@10", 0))
best = ablation[best_key]
print(f"Analyzing errors for: {best['label']}")

pq_df = pd.DataFrame(best["per_query"]).sort_values("hits@10")

zero_hit = pq_df[pq_df["hits@10"] == 0]
print(f"\nQueries with 0 hits@10: {len(zero_hit)} / {len(pq_df)}")
print("-" * 80)
for _, row in zero_hit.head(15).iterrows():
    print(f"  [{row['query_id'][:8]}] golden={row['golden_count']} | {row['query'][:80]}")

print(f"\nHits@10 distribution:")
print(pq_df["hits@10"].describe())

Analyzing errors for: 4. Dense + BM25 + Reranker

Queries with 0 hits@10: 112 / 291
--------------------------------------------------------------------------------
  [2f20a654] golden=1 | SERVICE_UNAVAILABLE error for CSAT on Ticket Snap-in
  [7e8b3140] golden=4 | where to find agent to test
  [9a3f3a87] golden=19 | fields with options Any of or None of
  [f0383078] golden=1 | delete ticket without notifying customer
  [03b9886d] golden=45 | find issues assigned to particular team member
  [745722d9] golden=17 | build snap-kit based widgets on plug
  [2558f7d0] golden=4 | auto-assign tickets to available agents
  [472c4905] golden=4 | update multiple tickets status in DevRev without going through individually
  [19959f02] golden=1 | Show Sentiment for Ticket as a Column
  [aabf6932] golden=1 | turn off suggestions like 'let me rephrase below
  [e80f2dd2] golden=21 | check schema of subtype of a ticket
  [7a7aed97] golden=17 | enable omnipresent agent experience within the app
  [89bec

In [28]:
# Deep dive: where do golden docs rank for a zero-hit query?
if len(zero_hit) > 0:
    sample = zero_hit.iloc[0]
    q = sample["query"]
    q_id = sample["query_id"]

    golden = set()
    for item in annotated_queries:
        if item["query_id"] == q_id:
            golden = {r["id"] for r in item["retrievals"]}
            break

    print(f"Query: {q}")
    print(f"Golden IDs: {golden}")

    q_emb = embed_single(q)
    dense_all = search_dense(q_emb, k=500)
    bm25_all = search_bm25(q, k=500)

    dense_rank = {doc_ids[idx]: rank for rank, (idx, _) in enumerate(dense_all)}
    bm25_rank = {doc_ids[idx]: rank for rank, (idx, _) in enumerate(bm25_all)}

    print(f"\n{'Golden ID':<40} {'Dense Rank':>12} {'BM25 Rank':>12}")
    print("-" * 66)
    for gid in golden:
        dr = dense_rank.get(gid, ">500")
        br = bm25_rank.get(gid, ">500")
        print(f"  {gid:<38} {str(dr):>12} {str(br):>12}")

Query: SERVICE_UNAVAILABLE error for CSAT on Ticket Snap-in
Golden IDs: {'ART-1174_KNOWLEDGE_NODE-4'}

Golden ID                                  Dense Rank    BM25 Rank
------------------------------------------------------------------
  ART-1174_KNOWLEDGE_NODE-4                       403         >500


## 9. Generate Test Query Submissions

In [29]:
TOP_K = 10
test_results = []

for item in tqdm(test_queries, desc="Generating test submissions"):
    query = item["query"]
    ranked = pipe_hybrid_rerank(query, top_k=TOP_K)

    retrievals = []
    for idx, score in ranked:
        retrievals.append({
            "id": doc_ids[idx],
            "text": doc_texts[idx],
            "title": doc_titles[idx],
        })

    test_results.append({
        "query_id": item["query_id"],
        "query": query,
        "retrievals": retrievals,
    })

print(f"Generated {len(test_results)} test results")

Generating test submissions: 100%|██████████| 92/92 [04:59<00:00,  3.26s/it]

Generated 92 test results


In [30]:
# Save results
OUTPUT_FILE = "test_queries_results.json"

with open(OUTPUT_FILE, "w") as f:
    json.dump(test_results, f, indent=2)
print(f"Saved {len(test_results)} results to {OUTPUT_FILE}")

# Save ablation metrics
eval_output = {k: {kk: vv for kk, vv in v.items() if kk != 'per_query'}
               for k, v in ablation.items()}
with open("ablation_metrics.json", "w") as f:
    json.dump(eval_output, f, indent=2)
print("Saved ablation_metrics.json")

# Preview
print(f"\nSample result:")
print(json.dumps(test_results[0], indent=2, default=str)[:800])

Saved 92 results to test_queries_results.json
Saved ablation_metrics.json

Sample result:
{
  "query_id": "a97f93d2-410a-431f-ae9a-1e23ed35d74c",
  "query": "end customer organization name not appearing in ticket or conversation",
  "retrievals": [
    {
      "id": "ART-1981_KNOWLEDGE_NODE-30",
      "text": "conversation of which you are not the owner, let the owner know to respond. It's beneficial to retain the same point of contact for the duration of the conversation unless the owner refers some another user.\\n* If the conversation has a customer org that's unidentified or is new, add yourself (the customer experience engineer) as the owner of the ticket. Try to find the appropriate owner for the customer org and update the customer record accordingly.\\n* Change the stage of the conversation to",
      "title": "Support best practices | Computer for Support Teams | DevRe


## 10. Parameter Tuning

| Parameter | Default | Range | Effect |
|-----------|---------|-------|--------|
| `dense_k` | 100 | 50-300 | Dense candidates for RRF pool |
| `bm25_k` | 100 | 50-300 | BM25 candidates for RRF pool |
| RRF `k` | 60 | 10-100 | Lower = more weight to top ranks |
| Rerank input | 100 | 50-200 | Candidates for reranker |
| Doc truncation | 512 | 256-1024 | More context vs speed |

In [32]:
def pipe_tuned(query, top_k=50, dense_k=150, bm25_k=150, rrf_k=60, rerank_n=100, doc_trunc=512):
    q_emb = embed_single(query)
    dense = search_dense(q_emb, k=dense_k)
    sparse = search_bm25(query, k=bm25_k)
    fused = rrf_fuse(dense, sparse, k=rrf_k)
    cands = fused[:rerank_n]
    pairs = [(query, documents[idx][:doc_trunc]) for idx, _ in cands]
    scores = reranker.predict(pairs, show_progress_bar=False)
    scored = [(cands[i][0], float(scores[i])) for i in range(len(cands))]
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]


TUNE_QUERIES = 50

configs = [
    {"dense_k": 100, "bm25_k": 100, "rrf_k": 60, "rerank_n": 100, "doc_trunc": 512, "label": "baseline"},
    {"dense_k": 200, "bm25_k": 200, "rrf_k": 60, "rerank_n": 150, "doc_trunc": 512, "label": "wider pool"},
    {"dense_k": 100, "bm25_k": 100, "rrf_k": 30, "rerank_n": 100, "doc_trunc": 512, "label": "rrf_k=30"},
    {"dense_k": 150, "bm25_k": 150, "rrf_k": 60, "rerank_n": 100, "doc_trunc": 768, "label": "longer docs"},
]

tune_results = []
for cfg in configs:
    label = cfg.pop("label")
    fn = lambda q, top_k=50, _c=cfg: pipe_tuned(q, top_k=top_k, **_c)
    m = evaluate_pipeline(fn, annotated_queries, max_queries=TUNE_QUERIES, label=label)
    tune_results.append(m)
    cfg["label"] = label

print("\n" + "=" * 80)
print("TUNING RESULTS (subset)")
print("=" * 80)
print(f"{'Config':<20} {'R@10':>8} {'R@50':>8} {'P@10':>8}")
print("-" * 50)
for m in tune_results:
    print(f"{m['label']:<20} {m['recall@10']:>7.2f}% {m['recall@50']:>7.2f}% {m['precision@10']:>7.2f}%")

Eval: baseline: 100%|██████████| 50/50 [01:55<00:00,  2.32s/it]



  baseline -- 50 queries
  Recall@10: 20.08%   Precision@10: 13.60%   MRR@10: 0.4072
  Recall@50: 27.88%   Precision@50: 4.44%   MRR@50: 0.4132


Eval: wider pool: 100%|██████████| 50/50 [04:23<00:00,  5.27s/it]



  wider pool -- 50 queries
  Recall@10: 21.36%   Precision@10: 14.60%   MRR@10: 0.4177
  Recall@50: 28.68%   Precision@50: 4.68%   MRR@50: 0.4220


Eval: rrf_k=30: 100%|██████████| 50/50 [02:55<00:00,  3.51s/it]



  rrf_k=30 -- 50 queries
  Recall@10: 20.08%   Precision@10: 13.60%   MRR@10: 0.4072
  Recall@50: 27.88%   Precision@50: 4.44%   MRR@50: 0.4132


Eval: longer docs: 100%|██████████| 50/50 [03:15<00:00,  3.91s/it]


  longer docs -- 50 queries
  Recall@10: 21.23%   Precision@10: 14.20%   MRR@10: 0.4247
  Recall@50: 28.09%   Precision@50: 4.52%   MRR@50: 0.4279

TUNING RESULTS (subset)
Config                   R@10     R@50     P@10
--------------------------------------------------
baseline               20.08%   27.88%   13.60%
wider pool             21.36%   28.68%   14.60%
rrf_k=30               20.08%   27.88%   13.60%
longer docs            21.23%   28.09%   14.20%
